# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**:

**ID**:

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [2]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `c:\Users\Donny\OneDrive\Post-Grad\Courses\BEE 5750\Homework 5\hw5-dc-hw5`
   Installed Measures ─────────── v0.3.3
   Installed GR_jll ───────────── v0.73.18+0
   Installed OpenBLAS32_jll ───── v0.3.29+0
   Installed PlotUtils ────────── v1.4.4
   Installed OpenSSL ──────────── v1.6.0
   Installed StaticArrays ─────── v1.9.15
   Installed MutableArithmetics ─ v1.6.7
   Installed HiGHS_jll ────────── v1.12.0+0
   Installed FFMPEG ───────────── v0.4.5
   Installed StaticArraysCore ─── v1.4.4
   Installed Pango_jll ────────── v1.57.0+0
   Installed JSON ─────────────── v1.3.0
   Installed METIS_jll ────────── v5.1.3+0
   Installed GR ───────────────── v0.73.18
   Installed JuMP ─────────────── v1.29.3
   Installed DataStructures ───── v0.19.3
   Installed Graphs ───────────── v1.13.1
   Installed Interpolations ───── v0.16.2
   Installed Adapt ────────────── v4.4.0
   Installed StableRNGs ───────── v1.0.4
   Installed GraphRecipes ─────── v0.5.15
   Installed Chai

In [1]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

In [ ]:
# Initialize the Model
model = Model(HiGHS.Optimizer)

# SETS & PARAMETERS
# Cities (1, 2, 3) and Facilities (LF, MRF, WTE)
cities = 1:3
facilities = ["LF", "MRF", "WTE"]

# Waste Generation (Mg/day)
gen = [100, 90, 120]

# Capacities (Mg/day)
cap = Dict("LF" => 200, "MRF" => 350, "WTE" => 210)

# Fixed Costs ($/day)
fixed_cost = Dict("LF" => 2000, "MRF" => 1500, "WTE" => 2500)

# Tipping Costs ($/Mg)
tipping_cost = Dict("LF" => 50, "MRF" => 7, "WTE" => 60)

# Recycling Cost ($/Mg recycled)
recycle_proc_cost = 40 

# Process Parameters
r_overall = 0.3775     # Recycling rate
r_residual = 0.6225    # Residual rate (1 - r)
ash_raw = 0.1641       # Ash from raw waste
ash_res = 0.1386       # Ash from MRF residuals

# Transport Cost ($/Mg-km)
trans_rate = 1.5

# Distances (City to Facility)
# Rows: Cities 1, 2, 3. Cols: LF, MRF, WTE
dist_city_fac = [
    5  30 15;
    15 25 10;
    13 45 20
]

# Distances (Between Facilities)
dist_mrf_lf = 32
dist_mrf_wte = 15
dist_wte_lf = 18

# DECISION VARIABLES 
# x[i, j]: Waste from City i to Facility j
@variable(model, x[1:3, 1:3] >= 0) 

# y_flow: Specific facility-to-facility flows
@variable(model, y_mrf_lf >= 0)
@variable(model, y_mrf_wte >= 0)
@variable(model, y_wte_lf >= 0)

# z[j]: Binary status (1 if open, 0 if closed)
@variable(model, z[1:3], Bin) 

# OBJECTIVE FUNCTION
# 1. Transport Costs
transport_cost = sum(trans_rate * dist_city_fac[i,j] * x[i,j] for i in 1:3, j in 1:3) +
                 trans_rate * dist_mrf_lf * y_mrf_lf +
                 trans_rate * dist_mrf_wte * y_mrf_wte +
                 trans_rate * dist_wte_lf * y_wte_lf

# 2. Fixed Costs (Indices: 1=LF, 2=MRF, 3=WTE)
fac_fixed_cost = fixed_cost["LF"]*z[1] + fixed_cost["MRF"]*z[2] + fixed_cost["WTE"]*z[3]

# 3. Variable/Processing Costs
# LF Cost (Inputs: Direct x + MRF resid + WTE ash)
cost_lf = tipping_cost["LF"] * (sum(x[i,1] for i in 1:3) + y_mrf_lf + y_wte_lf)

# MRF Cost (Inputs: Direct x) + Recycling Processing Cost
# Note: Recycling cost applies to the recycled fraction (r_overall * input)
cost_mrf = tipping_cost["MRF"] * sum(x[i,2] for i in 1:3) + 
           recycle_proc_cost * (r_overall * sum(x[i,2] for i in 1:3))

# WTE Cost (Inputs: Direct x + MRF resid)
cost_wte = tipping_cost["WTE"] * (sum(x[i,3] for i in 1:3) + y_mrf_wte)

@objective(model, Min, transport_cost + fac_fixed_cost + cost_lf + cost_mrf + cost_wte)

# CONSTRAINTS

# 1. Waste Generation (Supply)
for i in 1:3
    @constraint(model, sum(x[i,j] for j in 1:3) == gen[i])
end

# 2. Mass Balance: MRF Residuals
# Output residuals = Input * (1 - recycling_rate)
@constraint(model, y_mrf_lf + y_mrf_wte == r_residual * sum(x[i,2] for i in 1:3))

# 3. Mass Balance: WTE Ash
# Ash = (Raw Input * Raw Ash Rate) + (Residual Input * Residual Ash Rate)
@constraint(model, y_wte_lf == ash_raw * sum(x[i,3] for i in 1:3) + ash_res * y_mrf_wte)

# 4. Capacity & Linking Constraints (Big M)
# LF (Index 1)
@constraint(model, sum(x[i,1] for i in 1:3) + y_mrf_lf + y_wte_lf <= cap["LF"] * z[1])

# MRF (Index 2)
@constraint(model, sum(x[i,2] for i in 1:3) <= cap["MRF"] * z[2])

# WTE (Index 3)
@constraint(model, sum(x[i,3] for i in 1:3) + y_mrf_wte <= cap["WTE"] * z[3])

# SOLVE
optimize!(model)

# REPORT
println("Status: ", termination_status(model))
println("Optimal Objective Cost: \$", objective_value(model))

println("\nFacility Status:")
println("LF Open? ", value(z[1]))
println("MRF Open? ", value(z[2]))
println("WTE Open? ", value(z[3]))

println("\nPrimary Flows (City -> Facility):")
for i in 1:3
    println("City $i -> LF: ", value(x[i,1]))
    println("City $i -> MRF: ", value(x[i,2]))
    println("City $i -> WTE: ", value(x[i,3]))
end

println("\nSecondary Flows:")
println("MRF -> LF: ", value(y_mrf_lf))
println("MRF -> WTE: ", value(y_mrf_wte))
println("WTE -> LF: ", value(y_wte_lf))

Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 8 rows; 15 cols; 34 nonzeros; 3 integer variables (3 binary)
Coefficient ranges:
  Matrix  [1e-01, 4e+02]
  Cost    [6e+01, 2e+03]
  Bound   [1e+00, 1e+00]
  RHS     [9e+01, 1e+02]
Presolving model
8 rows, 15 cols, 34 nonzeros  0s
7 rows, 13 cols, 31 nonzeros  0s
Presolve reductions: rows 7(-1); columns 13(-2); nonzeros 31(-3) 

Solving MIP model with:
   7 rows
   13 cols (2 binary, 0 integer, 0 implied int., 11 continuous, 0 domain fixed)
   31 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective 

Based on the JuMP solver above, we obtained that the optimal cost based on the Objective Functino is **$27855,48**. This is achieved by
1. City 1 send all of its waste to landfilil
2. City 2 send all of its waste to WTE
3. City 3 send 78,4 Mg/day of its waste to landfill and 41,6 Mg/day of its waste to WTE
4. MRF facility will not be operated

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

## References

LLM was incorporated in solving the problems in the following manner

1. Problem 1.5 - After building the decision variable, objective function, and constraints, LLM was incorporated to build the JuMP draft code

2. Problem 2.2 - LLM was used to consult on the constraints for the stochastic formulation
